# 01 - Bronze to Silver

This notebook reads from the Bronze OneLake shortcuts (pointing to `RDW - Open Data/RDWLake`) and writes cleansed data to the Silver schema.

**Transformations applied:**
- Column names → `snake_case`
- Date parsing (integer YYYYMMDD, string, epoch ms → date)
- Numeric casting (string → int/double)
- Deduplication by business key
- Overwrite mode for idempotency

In [ ]:
# Create the silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [ ]:
import re
from pyspark.sql.functions import (
    col, to_date, when, lit, from_unixtime, trim
)
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DateType


def standardize_columns(df):
    """Rename all columns to lowercase snake_case."""
    for c in df.columns:
        new_name = re.sub(r'[^a-zA-Z0-9]', '_', c).lower()
        new_name = re.sub(r'_+', '_', new_name).strip('_')
        if new_name != c:
            df = df.withColumnRenamed(c, new_name)
    return df


def parse_int_date(df, col_name, fmt="yyyyMMdd"):
    """Parse integer YYYYMMDD to date type."""
    return df.withColumn(col_name, to_date(col(col_name).cast("string"), fmt))


def parse_str_date(df, col_name, fmt="yyyyMMdd"):
    """Parse string date to date type."""
    return df.withColumn(col_name,
        when(col(col_name).isNotNull() & (trim(col(col_name)) != lit("")),
             to_date(col(col_name), fmt)
        ).otherwise(lit(None).cast(DateType()))
    )


def parse_epoch_ms_date(df, col_name):
    """Parse epoch milliseconds (long) to date type."""
    return df.withColumn(col_name,
        when(col(col_name).isNotNull(),
             from_unixtime(col(col_name) / 1000).cast(DateType())
        ).otherwise(lit(None).cast(DateType()))
    )


def safe_cast(df, col_name, target_type):
    """Safely cast a column; non-parseable values become null."""
    return df.withColumn(col_name, col(col_name).cast(target_type))


print("Helper functions defined.")

## Silver: vehicles
Source: `bronze.gekentekende_voertuigen` → `silver.vehicles`

In [ ]:
df_vehicles = spark.read.table("bronze.gekentekende_voertuigen")
print(f"Bronze vehicles: {df_vehicles.count()} rows, {len(df_vehicles.columns)} columns")

In [ ]:
# Standardize column names
df_vehicles = standardize_columns(df_vehicles)

# Parse date columns
# vervaldatum_apk is integer YYYYMMDD
df_vehicles = parse_int_date(df_vehicles, "vervaldatum_apk")
# registratie_datum_goedkeuring_afschrijvingsmoment_bpm is integer YYYYMMDD
if "registratie_datum_goedkeuring_afschrijvingsmoment_bpm" in df_vehicles.columns:
    df_vehicles = parse_int_date(df_vehicles, "registratie_datum_goedkeuring_afschrijvingsmoment_bpm")

# String date columns (YYYYMMDD format)
str_date_cols = [
    "datum_tenaamstelling", "datum_eerste_toelating",
    "datum_eerste_tenaamstelling_in_nederland",
    "vervaldatum_tachograaf"
]
for dc in str_date_cols:
    if dc in df_vehicles.columns:
        df_vehicles = parse_str_date(df_vehicles, dc)

# Drop the _dt duplicate date columns (they are string representations)
dt_cols = [c for c in df_vehicles.columns if c.endswith("_dt")]
df_vehicles = df_vehicles.drop(*dt_cols)

# Cast string numeric columns
int_cast_cols = ["bruto_bpm", "aantal_zitplaatsen", "aantal_passagiers_zitplaatsen_wettelijk"]
for ic in int_cast_cols:
    if ic in df_vehicles.columns:
        df_vehicles = safe_cast(df_vehicles, ic, IntegerType())

dbl_cast_cols = ["catalogusprijs", "lengte", "breedte", "wielbasis", "vermogen_massarijklaar"]
for dc in dbl_cast_cols:
    if dc in df_vehicles.columns:
        df_vehicles = safe_cast(df_vehicles, dc, DoubleType())

# Drop API link columns (not useful for analytics)
api_cols = [c for c in df_vehicles.columns if c.startswith("api_")]
df_vehicles = df_vehicles.drop(*api_cols)

# Deduplication by kenteken
df_vehicles = df_vehicles.dropDuplicates(["kenteken"])

# Write to silver
df_vehicles.write.mode("overwrite").saveAsTable("silver.vehicles")
print(f"silver.vehicles written: {spark.read.table('silver.vehicles').count()} rows")

## Silver: vehicle_fuels
Source: `bronze.gekentekende_voertuigen_brandstof` → `silver.vehicle_fuels`

In [ ]:
df_fuels = spark.read.table("bronze.gekentekende_voertuigen_brandstof")
print(f"Bronze vehicle_fuels: {df_fuels.count()} rows, {len(df_fuels.columns)} columns")

# Standardize column names
df_fuels = standardize_columns(df_fuels)

# Cast string numeric columns to proper types
str_to_dbl = [
    "brandstof_verbruik_gecombineerd_wltp",
    "brandstof_verbruik_gewogen_gecombineerd_wltp",
    "elektrisch_verbruik_enkel_elektrisch_wltp",
    "elektrisch_verbruik_extern_opladen_wltp",
    "max_vermogen_15_minuten", "max_vermogen_60_minuten",
    "netto_max_vermogen_elektrisch",
    "opgegeven_maximum_snelheid",
    "brandstofverbruik_gewogen_gecombineerd",
    "elektriciteitsverbruik_gewogen_gecombineerd",
    "emissie_deeltjes_type1_wltp"
]
for c in str_to_dbl:
    if c in df_fuels.columns:
        df_fuels = safe_cast(df_fuels, c, DoubleType())

# Deduplication by kenteken + brandstof_volgnummer
df_fuels = df_fuels.dropDuplicates(["kenteken", "brandstof_volgnummer"])

# Write to silver
df_fuels.write.mode("overwrite").saveAsTable("silver.vehicle_fuels")
print(f"silver.vehicle_fuels written: {spark.read.table('silver.vehicle_fuels').count()} rows")

## Silver: fuels_by_postal_code
Source: `bronze.brandstoffen_op_pc4` → `silver.fuels_by_postal_code`

In [ ]:
df_pc4 = spark.read.table("bronze.brandstoffen_op_pc4")
print(f"Bronze fuels by PC4: {df_pc4.count()} rows")

# Standardize column names
df_pc4 = standardize_columns(df_pc4)

# Cast extern_oplaadbaar to boolean
df_pc4 = df_pc4.withColumn("extern_oplaadbaar",
    when(col("extern_oplaadbaar") == "Ja", lit(True))
    .when(col("extern_oplaadbaar") == "Nee", lit(False))
    .otherwise(lit(None).cast(BooleanType()))
)

# Write to silver
df_pc4.write.mode("overwrite").saveAsTable("silver.fuels_by_postal_code")
print(f"silver.fuels_by_postal_code written: {spark.read.table('silver.fuels_by_postal_code').count()} rows")

## Silver: parking_addresses
Source: `bronze.parkeeradres` → `silver.parking_addresses`

In [ ]:
df_park_addr = spark.read.table("bronze.parkeeradres")
print(f"Bronze parking addresses: {df_park_addr.count()} rows")

# Standardize column names
df_park_addr = standardize_columns(df_park_addr)

# Write to silver
df_park_addr.write.mode("overwrite").saveAsTable("silver.parking_addresses")
print(f"silver.parking_addresses written: {spark.read.table('silver.parking_addresses').count()} rows")

## Silver: parking_area_specs
Source: `bronze.specificaties_parkeergebied` → `silver.parking_area_specs`

In [ ]:
df_park_specs = spark.read.table("bronze.specificaties_parkeergebied")
print(f"Bronze parking area specs: {df_park_specs.count()} rows")

# Standardize column names
df_park_specs = standardize_columns(df_park_specs)

# Convert epoch milliseconds to date
df_park_specs = parse_epoch_ms_date(df_park_specs, "startdatespecifications")
df_park_specs = parse_epoch_ms_date(df_park_specs, "enddatespecifications")

# Rename to cleaner names
df_park_specs = df_park_specs \
    .withColumnRenamed("startdatespecifications", "start_date") \
    .withColumnRenamed("enddatespecifications", "end_date") \
    .withColumnRenamed("areamanagerid", "area_manager_id") \
    .withColumnRenamed("areaid", "area_id") \
    .withColumnRenamed("chargingpointcapacity", "charging_point_capacity") \
    .withColumnRenamed("disabledaccess", "disabled_access") \
    .withColumnRenamed("maximumvehicleheight", "maximum_vehicle_height") \
    .withColumnRenamed("limitedaccess", "limited_access")

# Write to silver
df_park_specs.write.mode("overwrite").saveAsTable("silver.parking_area_specs")
print(f"silver.parking_area_specs written: {spark.read.table('silver.parking_area_specs').count()} rows")

## Summary
All 5 Silver tables have been created.

In [ ]:
silver_tables = ["vehicles", "vehicle_fuels", "fuels_by_postal_code", "parking_addresses", "parking_area_specs"]
for t in silver_tables:
    count = spark.read.table(f"silver.{t}").count()
    cols = len(spark.read.table(f"silver.{t}").columns)
    print(f"silver.{t}: {count:,} rows, {cols} columns")